# Probability, Conditional Probability and Bayes' Theorem

**Dataset:** `banking_operations.csv`  
**Tools:** pandas, NumPy, Matplotlib and topic-specific statistical/ML functions  

This notebook explains the concept in simple terms and connects every calculation to banking operations.

## 1. Core ideas

**Probability** measures how frequently an event occurs. **Conditional probability** measures an event after another condition is known. **Bayes' theorem** reverses a condition and updates a prior probability using new evidence.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

df = pd.read_csv("banking_operations.csv")
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"])

print("Dataset shape:", df.shape)
display(df.head())

## 2. Basic probabilities from transaction frequencies

Here, the empirical probability is the number of matching transactions divided by all transactions.

In [ ]:
status_counts = df["Status"].value_counts()
status_probability = df["Status"].value_counts(normalize=True).rename("Probability")
display(pd.concat([status_counts.rename("Count"), status_probability], axis=1))

p_failed = (df["Status"] == "Failed").mean()
p_pending = (df["Status"] == "Pending").mean()
print(f"P(Failed) = {p_failed:.3f}")
print(f"P(Pending) = {p_pending:.3f}")

## 3. Joint probability

A joint probability measures two events occurring together. Example: a transaction is both a Mobile Banking transaction and Pending.

In [ ]:
event_mobile_pending = (df["Channel"] == "Mobile Banking") & (df["Status"] == "Pending")
p_mobile_and_pending = event_mobile_pending.mean()
print(f"P(Mobile Banking and Pending) = {p_mobile_and_pending:.3f}")

joint_table = pd.crosstab(df["Channel"], df["Status"], normalize="all")
display(joint_table)

## 4. Conditional probability

`P(Pending | Mobile Banking)` asks: among Mobile Banking transactions only, what proportion is Pending?

In [ ]:
mobile = df[df["Channel"] == "Mobile Banking"]
p_pending_given_mobile = (mobile["Status"] == "Pending").mean()

pending = df[df["Status"] == "Pending"]
p_mobile_given_pending = (pending["Channel"] == "Mobile Banking").mean()

print(f"P(Pending | Mobile Banking) = {p_pending_given_mobile:.3f}")
print(f"P(Mobile Banking | Pending) = {p_mobile_given_pending:.3f}")
print("These two conditional probabilities are not generally equal.")

## 5. Bayes' theorem using pandas results

We calculate `P(Mobile | Pending)` from `P(Pending | Mobile)`, `P(Mobile)` and `P(Pending)`.

In [ ]:
p_mobile = (df["Channel"] == "Mobile Banking").mean()
p_pending = (df["Status"] == "Pending").mean()

p_mobile_given_pending_bayes = (
    p_pending_given_mobile * p_mobile / p_pending
)

bayes_steps = pd.Series({
    "P(Mobile Banking) - prior": p_mobile,
    "P(Pending | Mobile Banking) - likelihood": p_pending_given_mobile,
    "P(Pending) - evidence": p_pending,
    "P(Mobile Banking | Pending) - posterior": p_mobile_given_pending_bayes
})
display(bayes_steps.to_frame("Probability"))

## 6. Conditional-probability table by channel

Each row shows the status distribution within a channel. The rows sum to 1.

In [ ]:
conditional_status_by_channel = pd.crosstab(
    df["Channel"], df["Status"], normalize="index"
)
display(conditional_status_by_channel)

conditional_status_by_channel.plot(kind="bar", stacked=True, figsize=(9, 4), colormap="Blues")
plt.title("Conditional Probability of Status Given Channel")
plt.ylabel("Probability")
plt.legend(title="Status", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## Banking interpretation and cautions

A higher conditional failure or pending rate can help prioritize operational review. It does not prove that the channel causes the status. Small groups can produce unstable probabilities, and historical frequencies can change over time.